# သင်ခန်းစာ ၁၁ - ကိုယ်စားလှယ်မှ ကိုယ်စားလှယ်သို့ (A2A) ပရိုတိုကော


## ဆက်တင်


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A ပရိုတိုကောလ် ဆိုတာဘာလဲ?

The **Agent-to-Agent (A2A) protocol** သည် AI ကိုယ်စားလှယ်များအား တိုက်ရိုက်ဆက်သွယ်နိုင်ရန်၊
အချင်းချင်း ရှာဖွေနိုင်ရန်နှင့် ပူးပေါင်းဆောင်ရွက်နိုင်ရန် ခွင့်ပြုသော ဖွင့်လှစ် စံသတ်မှတ်ချက်တစ်ခုဖြစ်သည် — အဲဒါတွေကို အခြား framework များပေါ် မူတည် သို့မဟုတ် ဝန်ဆောင်မှုကွဲပြားမှုရှိသော အဆာဗာများပေါ် host ထားပေမယ့်ပါ အလုပ်လုပ်နိုင်စေသည်။

Key concepts:

- **ရှာဖွေရေး** – Agents publish an *ကိုယ်စားလှယ်ကတ်* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **သတင်းပို့ဆောင်ခြင်း** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **တာဝန် ဘဝအဆင့်များ** – A2A defines states such as *တင်သွင်းပြီး*, *လုပ်ဆောင်နေသည်*, *ပြီးမြောက်သွားသည်*, and
  *မအောင်မြင်* , giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## အထူးပြု ခရီးသွားအေဂျင်များ ဖန်တီးခြင်း


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Workflow ဖြင့် မျိုးစုံအေဂျင့် ပူးပေါင်းဆောင်ရွက်ခြင်း

ကျွန်ုပ်တို့သည် အဲဒီအေဂျင့် သုံးခုကို A2A မက်ဆေ့ချ် ပေးပို့ခြင်းကို ဆင်တူသော လိုက်တိုက်ဆက်လက်အလုပ်လုပ်သော လုပ်ငန်းစဉ်အဖြစ် ချိတ်ဆက်ထားသည်။

1. **CurrencyExchangeAgent** သည် အသုံးပြုသူ၏ တောင်းဆိုချက်ကို လက်ခံပြီး ငွေလဲလှယ်ဆိုင်ရာ လမ်းညွှန်ချက်များ ထုတ်ပေးသည်။
2. **ActivityPlannerAgent** သည် တိုးမြှင့်ထားသော နောက်ခံအချက်အလက်ကို လက်ခံကာ လှုပ်ရှားမှု အကြံပြုချက်များ ထပ်ဖြည့်ပေးသည်။
3. **TravelManagerAgent** သည် နှစ်ဖက် input များကို ပေါင်းစပ်၍ နောက်ဆုံး ခရီးစဉ် အကျဉ်းချုပ် တစ်ခု ပြုစုပေးသည်။


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ထုတ်လုပ်မှုတွင် A2A ကိုနားလည်ခြင်း

ထုတ်လုပ်မှုပတ်ဝန်းကျင်၌ A2A ပရိုတိုကောသည် ဝန်ဆောင်မှုများအကြား အင်အားကြီးသော အခြေအနေများကို ဖွင့်ပေးသည်။

 
| စွမ်းဆောင်ရည် | ဖော်ပြချက် |
|---|---|
| **Framework များအကြား အပြန်အလှန် ဆက်သွယ်နိုင်မှု** | တစ်ခုသော framework ဖြင့် တည်ဆောက်ထားသော agent တစ်ဦးသည် အခြား A2A ကိုက်ညီသော framework ဖြင့် တည်ဆောက်ထားသော agent များထံတွင် တာဝန်များကို ကိုယ်စားလှယ်ပေးနိုင်ပြီး အဖွဲ့အစည်းများအကြား မှန်ကန်သော ပူးပေါင်းနိုင်မှုကို အတည်ပြုစေသည်။ |
| **ဝန်ဆောင်မှုနယ်နိမိတ်များ** | Agents များသည် သီးခြားထားသော မိုက်ခရိုဆာဗစ်များ၊ ကလောင်ဒ်ဒေသများ သို့မဟုတ် ကွဲပြားနေသည့် အဖွဲ့အစည်းများ၌ သီးခြားနေရနိုင်ပြီး ထိရောက်စွာ ပူးပေါင်းဆောင်ရွက်နိုင်သည်။ |
| **ဒိုင်နမစ် ရှာဖွေမှု** | Orchestrator တစ်ခုသည် runtime တွင် Agent Card မှတ်ပုံတင်စာရင်းကို မေးမြန်း၍ ပေးအပ်ထားသော အပိုင်းဆိုင်ရာ တာဝန်အတွက် အကောင်းဆုံး သက်ဆိုင်သူကို ရှာဖွေနိုင်သည်။ |
| **Streaming နှင့် push အသိပေးချက်များ** | A2A သည် အချိန်နှင့်တပြေးညီ တိုးတက်မှု အဆင့်များအတွက် Server-Sent Events (SSE) ကို ထောက်ပံ့ပြီး ရေရှည် ဆောင်ရွက်ရမည့် တာဝန်များအတွက် push အသိပေးချက်များကိုလည်း ပံ့ပိုးသည်။ |

The workflow we built above is a simplified, in-process version of this pattern. တကယ့်
တပ်ဆင်မှုတွင် agent တစ်ဦးချင်းစီသည် HTTP endpoint ကို ဖော်ထုတ်ကာ Agent Card ကို ထုတ်ဝေပြီး
A2A JSON-RPC ပရိုတိုကောဖြင့် ဆက်သွယ်မည်ဖြစ်သည်။


## အနှစ်ချုပ်

ဒီသင်ခန်းစာတွင် သင်သင်ယူခဲ့သောအရာများ:

1. **A2A ပရိုတိုကော ဆိုတာဘာလဲ** — အေဂျင့်-အေဂျင့် ရှာဖွေခြင်း၊ မက်ဆေ့ခ်ျပို့ဆက်သွယ်ခြင်းနှင့် တာဝန်စီမံခန့်ခွဲခြင်းများအတွက် ဖွင့်လင်းစံနစ်တစ်ခု။
2. **ထူးခြားအင်အားရှိတဲ့ အေဂျင့်များကို ဘယ်လို ဖန်တီးမလဲ** — a Currency Exchange agent, an Activity Planner agent, and a Travel Manager orchestrator.
3. **Agents များကို workflow ထဲချိတ်ဆက်နည်း** — `WorkflowBuilder` ကို အသုံးပြု၍ အေဂျင့်များအကြား အစဉ်လိုက် မက်ဆေ့ခ်ျ ပို့ဆက်မှုကို မော်ဒယ်ဖန်တီးခြင်း။
4. **A2A ကို ထုတ်လုပ်ပတ်ဝန်းကျင်တွင် မည်သို့ အလုပ်လုပ်သည်** — framework မျိုးစုံနှင့် service မျိုးစုံအကြား ပူးပေါင်းဆောင်ရွက်မှုကို ဒိုင်နမစ်ရှာဖွေမှုနှင့် စီးဆင်းလျက်ရှိသော အပ်ဒိတ်များဖြင့် အကောင်အထည်ဖော်ပေးခြင်း။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
အသိပေးချက်:
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု Co-op Translator (https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှုပေါ်အာရုံစိုက်ကြပေမယ့် အလိုအလျောက်ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါရှိနိုင်ကြောင်း သတိထားပေးလိုပါသည်။ မူရင်းစာတမ်းကို မူလဘာသာစကားဖြင့် ရရှိသည့် အာဏာရှိသော အရင်းအမြစ်အဖြစ် သတ်မှတ်ထားသင့်ပါသည်။ အရေးကြီးသော အချက်အလက်များအတွက် ကောင်းမွန်ကျွမ်းကျင်သော လူ့ဘာသာပြန် ဝန်ဆောင်မှုကို အသုံးပြုရန် အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းကြောင့် ဖြစ်ပေါ်နိုင်သည့် နားလည်မှုမှားယွင်းခြင်းများ သို့မဟုတ် အမှားဖော်ပြချက်များအတွက် ကျွန်ုပ်တို့သည် တာဝန်မရှိပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
